# Transaction data transformation

This notebook transforms the cleaned transaction data into a dataset containing useful analytical features

## Objectives

- Create temporal features from the transaction timestamp
- Represent purchases and refunds with an appropriate sign
- Exclude declined transactions from executed financial amounts
- Practise vectorised pandas operations
- Prepare the transaction data for later table joins and customer analysis

In [1]:
from pathlib import Path 
import pandas as pd

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

project_root

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence')

In [3]:
clean_transactions_path = (  project_root / "data" / "processed" / "transactions_clean.csv")
clean_transactions_path

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence/data/processed/transactions_clean.csv')

In [4]:
clean_transactions_path.exists()

True

## Loading the cleaned transactions

The cleaned transaction dataset will be used as the starting point for feature creation

The timestamp column is parsed directly as a pandas datetime column

In [5]:
transactions = pd.read_csv(clean_transactions_path, parse_dates = ["timestamp"])

transactions.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
0,T0001,A001,M001,2025-03-01 08:15:00,34.80,EUR,Purchase,Completed
1,T0002,A003,M002,2025-04-01 10:42:00,4.20,EUR,Purchase,Completed
2,T0003,A005,M003,2025-05-01 19:10:00,18.99,EUR,Purchase,Completed
3,T0004,A006,M004,2025-06-01 21:35:00,249.90,EUR,Purchase,Completed
4,T0005,A008,M005,2025-08-01 12:05:00,62.40,EUR,Purchase,Completed


In [6]:
transactions.dtypes

transaction_id                 str
account_id                     str
merchant_id                    str
timestamp           datetime64[us]
amount                     float64
currency                       str
transaction_type               str
status                         str
dtype: object

In [7]:
transactions.shape

(52, 8)

In [8]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    52 non-null     str           
 1   account_id        52 non-null     str           
 2   merchant_id       52 non-null     str           
 3   timestamp         52 non-null     datetime64[us]
 4   amount            52 non-null     float64       
 5   currency          52 non-null     str           
 6   transaction_type  52 non-null     str           
 7   status            52 non-null     str           
dtypes: datetime64[us](1), float64(1), str(6)
memory usage: 3.4 KB


In [9]:
transactions.isna().sum()

transaction_id      0
account_id          0
merchant_id         0
timestamp           0
amount              0
currency            0
transaction_type    0
status              0
dtype: int64

In [10]:
# Copy created for the transformations

transactions_features = transactions.copy(deep = True)

In [11]:
transactions_features.shape

(52, 8)

## Temporal features

The timestamp contains several pieces of information. I extract them into separate columns so that they can be analysed more easily

In [12]:
transactions_features = transactions_features.assign(
    year=transactions_features["timestamp"].dt.year,
    month=transactions_features["timestamp"].dt.month,
    month_name=(
        transactions_features["timestamp"]
        .dt.month_name()
    ),
    weekday=(
        transactions_features["timestamp"]
        .dt.day_name()
    ),
    hour=transactions_features["timestamp"].dt.hour,
)

In [13]:
transactions_features[
    [
        "transaction_id",
        "timestamp",
        "year",
        "month",
        "month_name",
        "weekday",
        "hour",
    ]
].head()

,transaction_id,timestamp,year,month,month_name,weekday,hour
0,T0001,2025-03-01 08:15:00,2025,3,March,Saturday,8
1,T0002,2025-04-01 10:42:00,2025,4,April,Tuesday,10
2,T0003,2025-05-01 19:10:00,2025,5,May,Thursday,19
3,T0004,2025-06-01 21:35:00,2025,6,June,Sunday,21
4,T0005,2025-08-01 12:05:00,2025,8,August,Friday,12


## Financial features

The original amount is always positive; I create different representations depending on the financial meaning of the transaction

In [14]:
transactions_features["absolute_amount"] = (transactions_features["amount"].abs())

In [15]:
transactions_features[["transaction_id","amount","absolute_amount",]].head()

,transaction_id,amount,absolute_amount
0,T0001,34.80,34.80
1,T0002,4.20,4.20
2,T0003,18.99,18.99
3,T0004,249.90,249.90
4,T0005,62.40,62.40


In [16]:
# A purchase represents an outflow of money and a return represents the opposite movement
transaction_sign = {
    "Purchase": 1,
    "Refund": -1
}

In [17]:
transactions_features["transaction_sign"] = (transactions_features["transaction_type"].map(transaction_sign))

In [18]:
transactions_features[
    [
        "transaction_id",
        "transaction_type",
        "transaction_sign",
    ]
].tail()

,transaction_id,transaction_type,transaction_sign
47,T0056,Purchase,1
48,T0057,Purchase,1
49,T0058,Purchase,1
50,T0059,Purchase,1
51,T0060,Refund,-1


In [19]:
transactions_features[
    "transaction_sign"
].isna().sum()

np.int64(0)

In [21]:
# Vectorized operation to create the amount with the sign

transactions_features["signed_amount"] = (transactions_features["absolute_amount"] 
                                        * transactions_features["transaction_sign"])


In [23]:
transactions_features.loc[transactions_features["transaction_type"].isin(
    ["Purchase","Refund"]
),
["transaction_id", "transaction_type", "absolute_amount","transaction_sign","signed_amount"]
].tail()

,transaction_id,transaction_type,absolute_amount,transaction_sign,signed_amount
47,T0056,Purchase,29.70,1,29.70
48,T0057,Purchase,5.10,1,5.10
49,T0058,Purchase,76.00,1,76.00
50,T0059,Purchase,459.00,1,459.00
51,T0060,Refund,19.99,-1,-19.99


In [24]:
# Exclusion of declined transactions

transactions_features["net_amount"] = (transactions_features["signed_amount"]
                                       .where(
                                           transactions_features["status"] == "Completed",
                                           0.0
                                       ))

In [25]:
transactions_features.loc[
    transactions_features["status"] == "Declined",
    [
        "transaction_id",
        "status",
        "signed_amount",
        "net_amount",
    ],
]

,transaction_id,status,signed_amount,net_amount
13,T0016,Declined,27.1,0.0
30,T0035,Declined,110.0,0.0
50,T0059,Declined,459.0,0.0


In [26]:
transactions_features[
    [
        "transaction_id",
        "transaction_type",
        "status",
        "amount",
        "absolute_amount",
        "signed_amount",
        "net_amount",
    ]
].tail(10)

,transaction_id,transaction_type,status,amount,absolute_amount,signed_amount,net_amount
42,T0049,Purchase,Completed,52.00,52.00,52.00,52.00
43,T0050,Purchase,Completed,130.60,130.60,130.60,130.60
44,T0052,Purchase,Completed,88.80,88.80,88.80,88.80
45,T0053,Purchase,Completed,625.00,625.00,625.00,625.00
46,T0055,Purchase,Completed,44.99,44.99,44.99,44.99
47,T0056,Purchase,Completed,29.70,29.70,29.70,29.70
48,T0057,Purchase,Completed,5.10,5.10,5.10,5.10
49,T0058,Purchase,Completed,76.00,76.00,76.00,76.00
50,T0059,Purchase,Declined,459.00,459.00,459.00,0.00
51,T0060,Refund,Completed,19.99,19.99,-19.99,-19.99


In [27]:
print("Raw amount total:", transactions_features["amount"].sum())
print("Executed net amount:", transactions_features["net_amount"].sum())

Raw amount total: 6306.31
Executed net amount: 5670.2300000000005


In [28]:
# Checks
transactions_features["year"].isna().sum()

np.int64(0)

In [29]:
transactions_features[
    "transaction_sign"
].notna().all()

np.True_

In [30]:
(
    transactions_features.loc[
        transactions_features["status"] == "Declined",
        "net_amount",
    ]
    == 0
).all()

np.True_

In [31]:
(
    transactions_features.loc[
        transactions_features["status"] == "Completed",
        "net_amount",
    ]
    == transactions_features.loc[
        transactions_features["status"] == "Completed",
        "signed_amount",
    ]
).all()

np.True_

## Initial transformation findings

- The timestamp was divided into year, month, month name, weekday and hour
- The absolute amount represents the size of each transaction
- Purchases were assigned a positive sign
- Refunds were assigned a negative sign
- Declined transactions were assigned a net financial effect of zero
- The transformations were performed using vectorised pandas operations

## Categorical transformations

I create analytical categories that describe the meaning, timing and size of each transaction

In [ ]:
# Difference between inflows and outflows
transactions_features["transaction_direction"] = transactions_features["transaction_type"].replace(
    {
        "Purchase": "Outflow",
        "Refund": "Inflow"
    }
)

In [34]:
transactions_features[
    [
        "transaction_id",
        "transaction_type",
        "transaction_direction",
    ]
].tail()

,transaction_id,transaction_type,transaction_direction
47,T0056,Purchase,Outflow
48,T0057,Purchase,Outflow
49,T0058,Purchase,Outflow
50,T0059,Purchase,Outflow
51,T0060,Refund,Inflow


In [35]:
transactions_features["transaction_direction"].value_counts()

transaction_direction
Outflow    51
Inflow      1
Name: count, dtype: int64

In [36]:
# Weekend indicator

transactions_features["is_weekend"] = (transactions_features["weekday"].isin(["Saturday", "Sunday"]))

In [37]:
transactions_features[
    [
        "transaction_id",
        "weekday",
        "is_weekend",
    ]
].head(10)

,transaction_id,weekday,is_weekend
0,T0001,Saturday,True
1,T0002,Tuesday,False
2,T0003,Thursday,False
3,T0004,Sunday,True
4,T0005,Friday,False
5,T0006,Monday,False
6,T0008,Saturday,True
7,T0009,Monday,False
8,T0010,Monday,False
9,T0011,Wednesday,False


In [38]:
transactions_features[
    "is_weekend"
].value_counts()

is_weekend
False    37
True     15
Name: count, dtype: int64

In [ ]:
# Difference between time periods
"""
00:00–05:59 Night
06:00–11:59 Morning
12:00–17:59 Afternoon
18:00–23:59 Evening
"""

time_bins = [-1, 5, 11, 17, 23]
time_labels = ["Night", "Morning", "Afternoon", "Evening"]

In [43]:
transactions_features["time_period"] = pd.cut(transactions_features["hour"], bins = time_bins, labels = time_labels)

In [44]:
transactions_features[
    [
        "transaction_id",
        "timestamp",
        "hour",
        "time_period",
    ]
].head(15)

,transaction_id,timestamp,hour,time_period
0,T0001,2025-03-01 08:15:00,8,Morning
1,T0002,2025-04-01 10:42:00,10,Morning
2,T0003,2025-05-01 19:10:00,19,Evening
3,T0004,2025-06-01 21:35:00,21,Evening
4,T0005,2025-08-01 12:05:00,12,Afternoon
5,T0006,2025-09-01 16:28:00,16,Afternoon
6,T0008,2025-11-01 09:20:00,9,Morning
7,T0009,2025-12-01 14:12:00,14,Afternoon
8,T0010,2025-01-13 20:05:00,20,Evening
9,T0011,2025-01-15 11:33:00,11,Morning


In [46]:
# Difference between transactions according to their amount
"""
Up to 25 Very small
> 25 - 100 Small
> 100 - 250	Medium
> 250 - 500	Large
> 500 Very large
"""

amount_bins = [0, 25, 100, 250, 500, float("inf")]
amount_labels = ["Very small","Small","Medium","Large","Very large"]

transactions_features["amount_band"] = pd.cut(transactions_features["absolute_amount"],
                                             bins = amount_bins, labels = amount_labels,
                                             include_lowest = True)



In [50]:
transactions_features[
    [
        "transaction_id",
        "absolute_amount",
        "amount_band",
    ]
].sort_values(
    "absolute_amount"
).head()

,transaction_id,absolute_amount,amount_band
36,T0042,3.9,Very small
1,T0002,4.2,Very small
48,T0057,5.1,Very small
18,T0022,5.6,Very small
14,T0017,6.8,Very small


In [51]:
transactions_features[
    [
        "transaction_id",
        "absolute_amount",
        "amount_band",
    ]
].sort_values(
    "absolute_amount",
    ascending = False
).head()

,transaction_id,absolute_amount,amount_band
37,T0043,899.0,Very large
45,T0053,625.0,Very large
28,T0033,540.0,Very large
50,T0059,459.0,Large
12,T0015,410.0,Large


In [52]:
transactions_features["amount_quartile"] = pd.qcut(transactions_features["absolute_amount"], q=4,
    labels=["Q1", "Q2","Q3","Q4"]
)

In [54]:
transactions_features[
    [
        "transaction_id",
        "absolute_amount",
        "amount_quartile",
    ]
].sort_values(
    "absolute_amount"
).head()

,transaction_id,absolute_amount,amount_quartile
36,T0042,3.9,Q1
1,T0002,4.2,Q1
48,T0057,5.1,Q1
18,T0022,5.6,Q1
14,T0017,6.8,Q1


In [57]:
transactions_features[
    "amount_quartile"
].value_counts(sort = False)

amount_quartile
Q1    14
Q2    12
Q3    13
Q4    13
Name: count, dtype: int64

## Estimated commission

For learning purposes, the project assumes:

- Completed purchases generate a 1.5% commission
- Declined transactions generate no commission
- Refunds generate no commission

<small>_This is a synthetic business rule and does not represent a real financial institution_</small>

In [58]:
# Base commission
transactions_features["commission_rate"] = 0.015

In [60]:
transactions_features["commission_rate"] = (transactions_features["commission_rate"]
                                            .mask(transactions_features["status"] == "Declined", 0.0))

In [61]:
transactions_features["commission_rate"] = (transactions_features["commission_rate"]
                                            .mask(transactions_features["transaction_type"] == "Refund",
                                                  0.0))

In [62]:
transactions_features["commission_amount"] = (
    transactions_features["absolute_amount"] * transactions_features["commission_rate"]
).round(2)

In [63]:
transactions_features[
    [
        "transaction_id",
        "transaction_type",
        "status",
        "absolute_amount",
        "commission_rate",
        "commission_amount",
    ]
].head()

,transaction_id,transaction_type,status,absolute_amount,commission_rate,commission_amount
0,T0001,Purchase,Completed,34.80,0.015,0.52
1,T0002,Purchase,Completed,4.20,0.015,0.06
2,T0003,Purchase,Completed,18.99,0.015,0.28
3,T0004,Purchase,Completed,249.90,0.015,3.75
4,T0005,Purchase,Completed,62.40,0.015,0.94


In [64]:
transactions_features[
    [
        "transaction_id",
        "timestamp",
        "weekday",
        "is_weekend",
        "hour",
        "time_period",
        "transaction_type",
        "transaction_direction",
        "absolute_amount",
        "amount_band",
        "amount_quartile",
        "commission_rate",
        "commission_amount",
        "net_amount",
    ]
].head(10)

,transaction_id,timestamp,weekday,is_weekend,hour,time_period,transaction_type,transaction_direction,absolute_amount,amount_band,amount_quartile,commission_rate,commission_amount,net_amount
0,T0001,2025-03-01 08:15:00,Saturday,True,8,Morning,Purchase,Outflow,34.80,Small,Q2,0.015,0.52,34.80
1,T0002,2025-04-01 10:42:00,Tuesday,False,10,Morning,Purchase,Outflow,4.20,Very small,Q1,0.015,0.06,4.20
2,T0003,2025-05-01 19:10:00,Thursday,False,19,Evening,Purchase,Outflow,18.99,Very small,Q1,0.015,0.28,18.99
3,T0004,2025-06-01 21:35:00,Sunday,True,21,Evening,Purchase,Outflow,249.90,Medium,Q4,0.015,3.75,249.90
4,T0005,2025-08-01 12:05:00,Friday,False,12,Afternoon,Purchase,Outflow,62.40,Small,Q3,0.015,0.94,62.40
5,T0006,2025-09-01 16:28:00,Monday,False,16,Afternoon,Purchase,Outflow,89.00,Small,Q3,0.015,1.34,89.00
6,T0008,2025-11-01 09:20:00,Saturday,True,9,Morning,Purchase,Outflow,320.00,Large,Q4,0.015,4.80,320.00
7,T0009,2025-12-01 14:12:00,Monday,False,14,Afternoon,Purchase,Outflow,74.50,Small,Q3,0.015,1.12,74.50
8,T0010,2025-01-13 20:05:00,Monday,False,20,Evening,Purchase,Outflow,13.00,Very small,Q1,0.015,0.20,13.00
9,T0011,2025-01-15 11:33:00,Wednesday,False,11,Morning,Purchase,Outflow,96.75,Small,Q3,0.015,1.45,96.75


In [65]:
transactions_features.shape

(52, 24)

# Transformation validation

Before saving the transformed dataset I verify that the new features follow the expected rules

In [66]:
feature_columns = ["year",
    "month",
    "month_name",
    "weekday",
    "hour",
    "absolute_amount",
    "transaction_sign",
    "signed_amount",
    "net_amount",
    "transaction_direction",
    "is_weekend",
    "time_period",
    "amount_band",
    "amount_quartile",
    "commission_rate",
    "commission_amount",
]

In [67]:
set(feature_columns).issubset(transactions_features.columns)

True

In [68]:
# Masks for the economic rules
completed_mask = (transactions_features["status"] == "Completed")
declined_mask = (transactions_features["status"] == "Declined")
completed_purchase_mask = (completed_mask & (transactions_features["transaction_type"] == "Purchase"))
zero_commission_mask = (declined_mask | (transactions_features["transaction_type"] == "Refund"))

In [69]:
# Creation of the validations
expected_purchase_commission = (transactions_features["absolute_amount"]*0.015).round(2)

In [71]:
# Validations summary
validation_results = pd.Series(
    {
        "row_count_preserved": (
            len(transactions_features)
            == len(transactions)
        ),
        "transaction_ids_are_unique": (
            transactions_features[
                "transaction_id"
            ].is_unique
        ),
        "all_feature_columns_exist": (
            set(feature_columns).issubset(
                transactions_features.columns
            )
        ),
        "features_have_no_missing_values": (
            transactions_features[
                feature_columns
            ]
            .isna()
            .sum()
            .sum()
            == 0
        ),
        "hours_are_valid": (
            transactions_features[
                "hour"
            ].between(0, 23).all()
        ),
        "absolute_amounts_are_positive": (
            transactions_features[
                "absolute_amount"
            ]
            .gt(0)
            .all()
        ),
        "declined_net_amount_is_zero": (
            transactions_features.loc[
                declined_mask,
                "net_amount",
            ]
            .eq(0)
            .all()
        ),
        "completed_net_matches_signed": (
            transactions_features.loc[
                completed_mask,
                "net_amount",
            ]
            .eq(
                transactions_features.loc[
                    completed_mask,
                    "signed_amount",
                ]
            )
            .all()
        ),
        "zero_commission_rules_hold": (
            transactions_features.loc[
                zero_commission_mask,
                "commission_amount",
            ]
            .eq(0)
            .all()
        ),
        "purchase_commission_is_correct": (
            transactions_features.loc[
                completed_purchase_mask,
                "commission_amount",
            ]
            .eq(
                expected_purchase_commission.loc[
                    completed_purchase_mask
                ]
            )
            .all()
        ),
    },
    name="passed",
)

validation_results.all()

np.True_

In [73]:
# Stats summary
transformation_summary = pd.Series(
    {
        "rows": len(transactions_features),
        "original_columns": len(
            transactions.columns
        ),
        "transformed_columns": len(
            transactions_features.columns
        ),
        "new_features": (
            len(transactions_features.columns)
            - len(transactions.columns)
        ),
        "weekend_transactions": int(
            transactions_features[
                "is_weekend"
            ].sum()
        ),
        "night_transactions": int(
            (
                transactions_features[
                    "time_period"
                ]
                == "Night"
            ).sum()
        ),
        "total_net_amount": (
            transactions_features[
                "net_amount"
            ].sum()
        ),
        "estimated_commission": (
            transactions_features[
                "commission_amount"
            ].sum()
        ),
    },
    name="value",
)

transformation_summary

rows                      52.00
original_columns           8.00
transformed_columns       24.00
new_features              16.00
weekend_transactions      15.00
night_transactions         0.00
total_net_amount        5670.23
estimated_commission      85.33
Name: value, dtype: float64

In [74]:
# Save dataset
processed_directory = (
    project_root
    / "data"
    / "processed"
)

transformed_transactions_path = (
    processed_directory
    / "transactions_features.csv"
)

In [75]:
transactions_features.to_csv(transformed_transactions_path, index = False)

In [76]:
transformed_transactions_path.exists()

True

In [77]:
saved_transactions_features = pd.read_csv(
    transformed_transactions_path,
    parse_dates=["timestamp"],
)

saved_transactions_features.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status,year,month,...,transaction_sign,signed_amount,net_amount,transaction_direction,is_weekend,time_period,amount_band,amount_quartile,commission_rate,commission_amount
0,T0001,A001,M001,2025-03-01 08:15:00,34.80,EUR,Purchase,Completed,2025,3,...,1,34.80,34.80,Outflow,True,Morning,Small,Q2,0.015,0.52
1,T0002,A003,M002,2025-04-01 10:42:00,4.20,EUR,Purchase,Completed,2025,4,...,1,4.20,4.20,Outflow,False,Morning,Very small,Q1,0.015,0.06
2,T0003,A005,M003,2025-05-01 19:10:00,18.99,EUR,Purchase,Completed,2025,5,...,1,18.99,18.99,Outflow,False,Evening,Very small,Q1,0.015,0.28
3,T0004,A006,M004,2025-06-01 21:35:00,249.90,EUR,Purchase,Completed,2025,6,...,1,249.90,249.90,Outflow,True,Evening,Medium,Q4,0.015,3.75
4,T0005,A008,M005,2025-08-01 12:05:00,62.40,EUR,Purchase,Completed,2025,8,...,1,62.40,62.40,Outflow,False,Afternoon,Small,Q3,0.015,0.94


In [79]:
saved_transactions_features.shape

(52, 24)

In [80]:
(
    saved_transactions_features.shape
    == transactions_features.shape
)

True

## Transformation results

The cleaned transactions were enriched with temporal, categorical and financial features

In the transformation process I created:

- Year, month, weekday and hour features
- Weekend and time period indicators
- Transaction direction
- Absolute, signed and executed net amounts
- Fixed amount bands
- Amount quartiles
- An estimated commission rate and amount

The difference between `pd.cut()` and `pd.qcut()` was examined, a row wise `apply()` solution was also compared with the vectorised `pd.cut()` solution. The vectorised result was retained

All validation checks passed, the original row count was preserved and the transformed dataset was saved to `data/processed/transactions_features.csv`